# Quark/gluon jet features and classification

This notebook continues from `demo_quark_gluon_samples.ipynb`. It:

1. reads the inclusive jet Parquet file and validates its schema;
2. compares physically motivated observables for truth-matched quark and gluon jets;
3. augments every jet with engineered features and writes a reusable Parquet dataset;
4. trains several classifiers with an event-grouped train/test split; and
5. compares their ROC performance and discusses which model is appropriate.

The main physics comparison uses **shape-only** inputs. A second BDT is deliberately given
jet kinematics to demonstrate how a classifier can exploit the unequal quark/gluon $p_T$
spectra rather than learning only substructure. Here `quark = 1` is the positive class.

Why not start with a transformer or autoencoder? This sample contains only a few thousand
labeled jets. A boosted decision tree is an excellent tabular-data baseline at this scale.
The constituent MLP below gives a lightweight raw-constituent comparison. Transformers become
more compelling with much larger samples; autoencoders are primarily useful for unsupervised
anomaly detection, not as a like-for-like supervised quark/gluon classifier.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 7
DATA_DIR = Path('data')
INPUT_PATH = DATA_DIR / 'inclusive_jets.parquet'
FEATURE_PATH = DATA_DIR / 'quark_gluon_jets_with_features.parquet'

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(RANDOM_STATE)

## A. Read and validate the jet data

The inclusive file is the source of truth: the quark and gluon samples are exact flavor
filters of it. List-valued constituent columns are read back as array-like objects by
PyArrow/Pandas.

In [ ]:
jets = pd.read_parquet(INPUT_PATH)

required_columns = {
    'event_id', 'jet_id', 'flavor', 'flavor_pdgid', 'matched_dr',
    'jet_pt', 'jet_eta', 'jet_phi', 'jet_mass', 'jet_e',
    'n_constituents', 'const_pt', 'const_eta', 'const_phi', 'const_pid',
}
missing = required_columns.difference(jets.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'
assert (jets['jet_pt'] > 20.0).all()
assert (jets['jet_eta'].abs() < 2.0).all()
assert not jets.duplicated(['event_id', 'jet_id']).any()

for column in ('const_pt', 'const_eta', 'const_phi', 'const_pid'):
    assert all(len(values) == count
               for values, count in zip(jets[column], jets['n_constituents']))

labeled = jets[jets['flavor'].isin(['quark', 'gluon'])].reset_index(drop=True)
print(f'Loaded {len(jets):,} accepted jets from {jets.event_id.nunique():,} events')
print(f'Truth-matched sample: {len(labeled):,} jets')
display(jets['flavor'].value_counts().rename('jets').to_frame())
display(labeled.head(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {'quark': 'steelblue', 'gluon': 'tomato'}

for flavor, color in colors.items():
    selected = labeled[labeled['flavor'] == flavor]
    axes[0].hist(selected['jet_pt'], bins=np.linspace(20, 150, 45), density=True,
                 histtype='step', linewidth=1.8, color=color, label=flavor)
    axes[1].hist(selected['jet_eta'], bins=np.linspace(-2, 2, 41), density=True,
                 histtype='step', linewidth=1.8, color=color, label=flavor)

axes[0].set(xlabel=r'Jet $p_T$ [GeV]', ylabel='Normalized density',
            title=r'Kinematics differ: $p_T$ can be a shortcut')
axes[1].set(xlabel=r'Jet $\eta$', ylabel='Normalized density',
            title='Accepted pseudorapidity')
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## B. Engineer interpretable jet-shape features

For constituent $i$, define $z_i=p_{T,i}/\sum_j p_{T,j}$ and
$\Delta R_i=\sqrt{(\eta_i-\eta_J)^2+(\phi_i-\phi_J)^2}$.

| Feature | Definition / interpretation |
|---|---|
| constituent multiplicity | Gluon jets tend to radiate more and have more particles |
| $m/p_T$ | Dimensionless jet mass; broader radiation raises the mass |
| $p_T^D$ | $\sqrt{\sum_i p_{T,i}^2}/\sum_i p_{T,i}$; large for hard fragmentation |
| leading fraction | $\max_i z_i$; another fragmentation-hardness measure |
| soft fraction | $\sum_{z_i<0.05} z_i$ |
| LHA | $\sum_i z_i\Delta R_i^{1/2}$, the Les Houches angularity |
| girth | $\sum_i z_i\Delta R_i$ |
| radial width | $\sqrt{\sum_i z_i\Delta R_i^2}$ |
| core fractions | fraction of constituent $p_T$ inside $\Delta R<0.1$ or $0.2$ |
| EEC-like moment | $\sum_{i<j}z_i z_j\Delta R_{ij}$ |

These are detector-level-style observables constructed from the stored constituents. The
truth matching quantities (`matched_dr`, `flavor_pdgid`) must never be classifier inputs.

In [ ]:
def wrap_phi(phi):
    return (phi + np.pi) % (2 * np.pi) - np.pi


def compute_jet_features(row):
    pt = np.asarray(row.const_pt, dtype=float)
    eta = np.asarray(row.const_eta, dtype=float)
    phi = np.asarray(row.const_phi, dtype=float)
    total_pt = pt.sum()

    if len(pt) == 0 or total_pt <= 0:
        return {
            'mass_over_pt': 0.0, 'ptd': 0.0, 'lead_pt_fraction': 0.0,
            'soft_pt_fraction': 0.0, 'angularity_beta_0p5': 0.0,
            'girth': 0.0, 'radial_width': 0.0,
            'core_pt_fraction_0p1': 0.0, 'core_pt_fraction_0p2': 0.0,
            'eec_beta_1': 0.0,
        }

    z = pt / total_pt
    deta = eta - row.jet_eta
    dphi = wrap_phi(phi - row.jet_phi)
    dr = np.hypot(deta, dphi)

    if len(pt) > 1:
        i, j = np.triu_indices(len(pt), k=1)
        pair_dr = np.hypot(eta[i] - eta[j], wrap_phi(phi[i] - phi[j]))
        eec_beta_1 = np.sum(z[i] * z[j] * pair_dr)
    else:
        eec_beta_1 = 0.0

    return {
        'mass_over_pt': row.jet_mass / row.jet_pt,
        'ptd': np.sqrt(np.sum(pt ** 2)) / total_pt,
        'lead_pt_fraction': z.max(),
        'soft_pt_fraction': z[z < 0.05].sum(),
        'angularity_beta_0p5': np.sum(z * np.sqrt(dr)),
        'girth': np.sum(z * dr),
        'radial_width': np.sqrt(np.sum(z * dr ** 2)),
        'core_pt_fraction_0p1': z[dr < 0.1].sum(),
        'core_pt_fraction_0p2': z[dr < 0.2].sum(),
        'eec_beta_1': eec_beta_1,
    }


feature_values = pd.DataFrame(
    [compute_jet_features(row) for row in tqdm(jets.itertuples(index=False), total=len(jets), desc='Computing jet features', unit='jet')]
)
jets_with_features = pd.concat(
    [jets.reset_index(drop=True), feature_values], axis=1
)
jets_with_features['abs_jet_eta'] = jets_with_features['jet_eta'].abs()

assert np.isfinite(feature_values.to_numpy()).all()
display(jets_with_features[['event_id', 'jet_id', 'flavor', 'n_constituents']
                           + list(feature_values.columns)].head())

## C. Save the augmented jet dataset

This file keeps the original jet and list-valued constituent columns and appends the new
features. It therefore supports both tabular models and later constituent-level models.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
jets_with_features.to_parquet(FEATURE_PATH, engine='pyarrow', index=False)
round_trip = pd.read_parquet(FEATURE_PATH)

assert len(round_trip) == len(jets_with_features)
assert list(round_trip.columns) == list(jets_with_features.columns)
for column in ('const_pt', 'const_eta', 'const_phi', 'const_pid'):
    assert all(len(values) == count
               for values, count in zip(round_trip[column], round_trip['n_constituents']))

print(f'Wrote {len(round_trip):,} jets and {len(round_trip.columns)} columns')
print(f'{FEATURE_PATH} ({FEATURE_PATH.stat().st_size / 1024**2:.2f} MiB)')

In [ ]:
plot_features = [
    ('n_constituents', 'Constituent multiplicity'),
    ('jet_mass', r'Jet mass [GeV]'),
    ('mass_over_pt', r'$m/p_T$'),
    ('ptd', r'$p_T^D$'),
    ('lead_pt_fraction', 'Leading constituent fraction'),
    ('angularity_beta_0p5', 'Les Houches angularity'),
    ('girth', 'Girth'),
    ('radial_width', 'Radial width'),
    ('eec_beta_1', r'EEC-like $\beta=1$ moment'),
]

matched_features = jets_with_features[
    jets_with_features['flavor'].isin(['quark', 'gluon'])
]
fig, axes = plt.subplots(3, 3, figsize=(14, 11))

for ax, (feature, label) in zip(axes.flat, plot_features):
    values = matched_features[feature]
    low, high = values.quantile([0.002, 0.995])
    if high <= low:
        high = low + 1
    bins = np.linspace(low, high, 40)
    for flavor, color in colors.items():
        selected = matched_features.loc[matched_features['flavor'] == flavor, feature]
        ax.hist(selected, bins=bins, density=True, histtype='step', linewidth=1.7,
                color=color, label=flavor)
    ax.set(xlabel=label, ylabel='Normalized density')
    ax.legend(fontsize=8)

plt.suptitle('Quark and gluon jet observables', y=1.01, fontsize=15)
plt.tight_layout()
plt.savefig('demo_quark_gluon_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## D. Build leakage-safe ML samples

The split is by `event_id`, not by individual jet: every jet from one generated event stays
entirely in train or test. The training set is downsampled to equal quark/gluon counts so all
models see the same balanced sample. The untouched test set retains its natural mixture.

The top-constituent representation sorts constituents by $p_T$, keeps the first 12, and stores
$(z,\Delta\eta,\Delta\phi,\Delta R,\mathrm{mask})$. It is a compact, fixed-size proxy for a raw
constituent model—not permutation invariant and not as expressive as a transformer.

In [ ]:
shape_features = [
    'n_constituents', 'mass_over_pt', 'ptd', 'lead_pt_fraction',
    'soft_pt_fraction', 'angularity_beta_0p5', 'girth', 'radial_width',
    'core_pt_fraction_0p1', 'core_pt_fraction_0p2', 'eec_beta_1',
]
shape_plus_kinematics = shape_features + ['jet_pt', 'abs_jet_eta']

model_data = jets_with_features[
    jets_with_features['flavor'].isin(['quark', 'gluon'])
].reset_index(drop=True)
model_data['label'] = (model_data['flavor'] == 'quark').astype(int)

event_ids = model_data['event_id'].unique()
train_events, test_events = train_test_split(
    event_ids, test_size=0.25, random_state=RANDOM_STATE
)
train_mask = model_data['event_id'].isin(train_events).to_numpy()
test_mask = model_data['event_id'].isin(test_events).to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
y_all = model_data['label'].to_numpy()
train_by_class = [np.flatnonzero(train_mask & (y_all == label)) for label in (0, 1)]
n_balanced = min(map(len, train_by_class))
train_indices = np.concatenate([
    rng.choice(indices, size=n_balanced, replace=False) for indices in train_by_class
])
rng.shuffle(train_indices)
test_indices = np.flatnonzero(test_mask)

assert set(model_data.loc[train_indices, 'event_id']).isdisjoint(
    set(model_data.loc[test_indices, 'event_id'])
)
print(f'Balanced training sample: {len(train_indices):,} jets '
      f'({n_balanced:,} per class)')
print(f'Natural test sample: {len(test_indices):,} jets')
display(model_data.loc[test_indices, 'flavor'].value_counts().rename('test jets').to_frame())

In [ ]:
MAX_CONSTITUENTS = 12


def top_constituent_inputs(row, max_constituents=MAX_CONSTITUENTS):
    pt = np.asarray(row.const_pt, dtype=float)
    eta = np.asarray(row.const_eta, dtype=float)
    phi = np.asarray(row.const_phi, dtype=float)
    order = np.argsort(-pt)[:max_constituents]
    pt, eta, phi = pt[order], eta[order], phi[order]

    result = np.zeros((max_constituents, 5), dtype=np.float32)
    if len(pt):
        z = pt / pt.sum()
        deta = eta - row.jet_eta
        dphi = wrap_phi(phi - row.jet_phi)
        n = len(pt)
        result[:n, 0] = z
        result[:n, 1] = deta
        result[:n, 2] = dphi
        result[:n, 3] = np.hypot(deta, dphi)
        result[:n, 4] = 1.0
    return result.ravel()


X_constituents = np.vstack([
    top_constituent_inputs(row) for row in tqdm(model_data.itertuples(index=False), total=len(model_data), desc='Building constituent inputs', unit='jet')
])
print('Top-constituent input shape:', X_constituents.shape)

## E. Train four controlled baselines

- **Logistic (shapes):** tests how much separation is approximately linear.
- **BDT (shapes):** the recommended small-data tabular baseline.
- **BDT (shapes + kinematics):** diagnostic model; improved scores may be spectrum learning.
- **Top-12 constituent MLP:** learns from a padded low-level representation without explicit
  angularities. This is closer in spirit to a transformer, but much lighter.

In [ ]:
X_shapes = model_data[shape_features].to_numpy(dtype=float)
X_shapes_kin = model_data[shape_plus_kinematics].to_numpy(dtype=float)

model_specs = {
    'Logistic (shapes)': (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        X_shapes,
    ),
    'BDT (shapes)': (
        GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.04, max_depth=2,
            subsample=0.8, random_state=RANDOM_STATE,
        ),
        X_shapes,
    ),
    'BDT (shapes + kinematics)': (
        GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.04, max_depth=2,
            subsample=0.8, random_state=RANDOM_STATE,
        ),
        X_shapes_kin,
    ),
    'Top-12 constituent MLP': (
        make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(64, 32), activation='relu',
                early_stopping=True, validation_fraction=0.2,
                max_iter=300, random_state=RANDOM_STATE,
            ),
        ),
        X_constituents,
    ),
}

fitted_models = {}
test_scores = {}
for name, (model, inputs) in model_specs.items():
    print(f'Training {name} ...')
    model.fit(inputs[train_indices], y_all[train_indices])
    fitted_models[name] = model
    test_scores[name] = model.predict_proba(inputs[test_indices])[:, 1]

print('Training complete.')

In [ ]:
def rejection_at_quark_efficiency(y_true, score, target_efficiency=0.5):
    quark_scores = score[y_true == 1]
    threshold = np.quantile(quark_scores, 1.0 - target_efficiency)
    gluon_acceptance = np.mean(score[y_true == 0] >= threshold)
    return np.inf if gluon_acceptance == 0 else 1.0 / gluon_acceptance


y_test = y_all[test_indices]
metrics = []
for name, score in test_scores.items():
    prediction = (score >= 0.5).astype(int)
    metrics.append({
        'model': name,
        'ROC AUC': roc_auc_score(y_test, score),
        'accuracy': accuracy_score(y_test, prediction),
        'balanced accuracy': balanced_accuracy_score(y_test, prediction),
        'gluon rejection @ 50% quark efficiency':
            rejection_at_quark_efficiency(y_test, score, 0.5),
    })

metrics_df = pd.DataFrame(metrics).sort_values('ROC AUC', ascending=False)
display(metrics_df.style.format({
    'ROC AUC': '{:.3f}',
    'accuracy': '{:.3f}',
    'balanced accuracy': '{:.3f}',
    'gluon rejection @ 50% quark efficiency': '{:.2f}',
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, score in test_scores.items():
    fpr, tpr, _ = roc_curve(y_test, score)
    auc = roc_auc_score(y_test, score)
    axes[0].plot(tpr, 1.0 / np.clip(fpr, 1e-3, None), linewidth=1.8,
                 label=f'{name} (AUC={auc:.3f})')
axes[0].set(
    xlabel='Quark efficiency', ylabel='Gluon rejection',
    title='Classifier performance', yscale='log',
    xlim=(0.05, 1.0), ylim=(1.0, 1e3),
)
axes[0].legend(fontsize=8)

ordered = metrics_df.sort_values('ROC AUC')
axes[1].barh(ordered['model'], ordered['ROC AUC'], color='slateblue')
axes[1].set(xlabel='ROC AUC', title='One-number comparison', xlim=(0.5, 1.0))
for y, value in enumerate(ordered['ROC AUC']):
    axes[1].text(value + 0.005, y, f'{value:.3f}', va='center')

plt.tight_layout()
plt.savefig('demo_quark_gluon_classifier_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
shape_bdt = fitted_models['BDT (shapes)']
importance = pd.Series(
    shape_bdt.feature_importances_, index=shape_features, name='importance'
).sort_values()

ax = importance.plot.barh(figsize=(7, 5), color='darkcyan')
ax.set(xlabel='BDT impurity-based importance', title='Which engineered features matter?')
plt.tight_layout()
plt.show()
display(importance.sort_values(ascending=False).to_frame())

## F. Which classifier is better?

Use the measured table above, not architecture fashion, to answer this:

- **For this dataset and an interpretable first result:** prefer the shape-only BDT if it has
  the best or near-best held-out AUC. It matches the small-sample, tabular setting and exposes
  feature importance.
- **Do not automatically select the kinematics-augmented winner.** If its advantage is large,
  it may mostly be exploiting the generator's unequal quark/gluon $p_T$ spectra. Reweight or
  bin in $p_T$ before claiming a substructure improvement.
- **The constituent MLP is a representation test.** If it lags the BDT, that does not prove
  low-level learning is inferior; it may need more events or a permutation-aware architecture.
- **A transformer / Particle Transformer** is a sensible next experiment after producing at
  least $\mathcal{O}(10^5)$ labeled jets and defining padded masks, normalization, validation,
  and compute budgets. Compare it on the exact same event split.
- **An autoencoder** is appropriate when training on a reference class and searching for
  anomalous jets using reconstruction error. It is not the natural supervised baseline here.

Accuracy alone is misleading for the naturally imbalanced test sample. ROC AUC and gluon
rejection at fixed quark efficiency are the more useful headline metrics.

In [ ]:
best_overall = metrics_df.iloc[0]
shape_rows = metrics_df[metrics_df['model'].isin(['Logistic (shapes)', 'BDT (shapes)'])]
best_shape = shape_rows.iloc[0]

print(f"Highest measured ROC AUC: {best_overall['model']} "
      f"({best_overall['ROC AUC']:.3f})")
print(f"Best strictly shape-based tabular model: {best_shape['model']} "
      f"({best_shape['ROC AUC']:.3f})")
print('Treat any gain from explicit jet_pt/eta inputs as a potential kinematic shortcut.')